# Spike Interface -- Spike Sorting 
## Pipeline notebook
### Loading in SpikeGlx recordings, applying preprocessing and/or drift correction, spike sorting with Kilosort, calculating quality metrics

--------------------------------------------------------------------------------------------------------------------------------------------

## **Importing packages and defining parameters**

This section of code sets up the file paths and configuration parameters for running spike sorting on neural data. It includes options for specifying the sorting algorithm, enabling drift correction, and customizing sorter parameters.

🔧 **Sorter Type:**  
[Sorters Module Documentation](https://spikeinterface.readthedocs.io/en/latest/modules/sorters.html)

The variable sorter_type specifies which spike sorting algorithm to use. Current supported options:
* `kilosort4` (default)

🔄 **Drift Correction:**  
[Motion Correction Module Documentation](https://spikeinterface.readthedocs.io/en/latest/how_to/handle_drift.html)

The drift_correct variable determines whether and how drift correction is applied. Available options:
* `None`           —  No drift correction applied during preprocessing, `do_correction` param in custom_params must also be set to False to completely disable drift correction 
* `dredge`         —  Uses the [DREDGe method](https://pubmed.ncbi.nlm.nih.gov/37961359/)
* `medicine`       -  Uses the [MEDiCINe method](https://www.eneuro.org/content/12/3/ENEURO.0529-24.2025)
* `rigid_fast`     -  Fast, but not very accurate method; uses center of mass + decentralized + inverse distance weighted
* `kilosort_like`  -  Consists of grid convolution + iterative_template + kriging, to mimic what is done in Kilosort

⚙️ **Other Parameters:**  
* `probe_id`: Selects the probe index (e.g., Neuropixels slot); default is 0.
* `toss_bad_chans`: Whether to exclude “bad” channels; default is True.
* `overwrite_preprocess`: If True, existing preprocessed data will be deleted and recreated.
* `overwrite_sorting`: If True, any existing sorting output will be re-run.

🗂 **Folder Naming Convention:**  
The names of output folders for preprocessed data and sorting results are automatically adjusted based on:
* The selected drift_correct method
* Any deviations from the default parameters for the specified sorter_type

This ensures that outputs from different sorting configurations are stored separately and clearly labeled.

In [4]:
# General packages
import numpy as np
from pathlib import Path
import os
import time
import shutil
import json

# Spike Interface packages
import spikeinterface.full as si
from spikeinterface.sortingcomponents.motion import correct_motion_on_peaks, interpolate_motion
from spikeinterface.sorters import run_sorter

global_job_kwargs = dict(n_jobs=40, chunk_duration='1s', progress_bar=True)
si.set_global_job_kwargs(**global_job_kwargs)

# Custom preprocessing/plotting functions
from si_utils import *

#os.environ['OPENBLAS_NUM_THREADS'] = '1'

In [5]:
# Define paths, required parameters 
raw_data_path  =  '/ix1/pmayo/lab_NHPdata/'  # parent path where data directories are located
session_name   =  'Ya_250515_s395_g0'  # full name of data directory

# Define inputs, optional parameters
probe_id        =   0           # probe index (if neuropixels this would be the imec slot), default = 0 
sorter_type     =  'kilosort4'  # sorter type, default = kilosort4
drift_correct   =  'dredge'     # drift correction type, default = 'none'

remove_bad_channels   =  True         # flag to remove "bad" channels, default = True

overwrite_preprocess  =  True  # if preprocessed data already exists then re-write it, default = False
overwrite_sorting     =  True  # if sorting already exists then re-sort it, default = False 

# Spike sorting parameters 
default_params = si.get_default_sorter_params(sorter_name_or_class=sorter_type)
print(f"--------------{sorter_type}--------------\n{default_params}\n------------------------------------------------------------------------------------------------\n")

custom_params = {
    'do_correction': False,
    'clear_cache': True
}
print(f"--------- custom parameters ---------\n{custom_params}\n------------------------------------------------------------------------------------------------")

--------------kilosort4--------------
{'fs': 30000, 'batch_size': 60000, 'nblocks': 1, 'Th_universal': 9, 'Th_learned': 8, 'nt': 61, 'shift': None, 'scale': None, 'artifact_threshold': inf, 'nskip': 25, 'whitening_range': 32, 'highpass_cutoff': 300, 'binning_depth': 5, 'sig_interp': 20, 'drift_smoothing': [0.5, 0.5, 0.5], 'nt0min': None, 'dmin': None, 'dminx': 32, 'min_template_size': 10, 'template_sizes': 5, 'nearest_chans': 10, 'nearest_templates': 100, 'max_channel_distance': 32, 'max_peels': 100, 'templates_from_data': True, 'n_templates': 6, 'n_pcs': 6, 'Th_single_ch': 6, 'acg_threshold': 0.2, 'ccg_threshold': 0.25, 'cluster_neighbors': 10, 'cluster_downsampling': 20, 'max_cluster_subset': None, 'x_centers': None, 'duplicate_spike_ms': 0.25, 'position_limit': 100, 'do_CAR': True, 'invert_sign': False, 'save_extra_vars': False, 'save_preprocessed_copy': False, 'torch_device': 'auto', 'bad_channels': None, 'clear_cache': False, 'do_correction': True, 'skip_kilosort_preprocessing': F

In [6]:
run_suffix = format_param_suffix(default_params, custom_params)
run_suffix = run_suffix or 'defaults'

data_folder, preprocess_folder, sorter_folder, metadata = make_folder_paths(raw_data_path, session_name, probe_id, drift_correct, sorter_type, run_suffix, overwrite_preprocess=overwrite_preprocess)
sorter_folder 

PosixPath('/ix1/pmayo/lab_NHPdata/Ya_250515_s395_g0/Ya_250515_s395_g0_imec0/kilosort4/dredge_cor0')

## **1. Preprocess the recording**
### **1.1**  Check whether preprocessed data is available: if so, load it; otherwise, load the raw recording instead.

In [ ]:
# If the preprocessed data already exists and overwrite_preprocess=False
if os.listdir(preprocess_folder):  
    print("Step 1.1: Loading preprocessed data...........................")
    PROC_RECORDING = si.load(preprocess_folder)

# If the preprocessed data does not exist, load raw recording
else: 
    print("Step 1.1: Loading raw data...........................")
    if metadata["probe_types"][probe_id] == "neuropixel":
        stream_names, stream_ids = si.get_neo_streams('spikeglx', data_folder)
        RAW_RECORDING = si.read_spikeglx(data_folder, stream_name=f'imec{probe_id}.ap', load_sync_channel=False)

    print("--------------------------------------------------")
    print("Sampling frequency:", RAW_RECORDING.get_sampling_frequency())
    print("Number of channels:", RAW_RECORDING.get_num_channels())
    print("Number of segments:", RAW_RECORDING.get_num_segments())
    print("Number of samples:", RAW_RECORDING.get_num_samples(segment_index=0))
    print("Duration of recording (min):", round((RAW_RECORDING.get_num_samples(segment_index=0)/RAW_RECORDING.get_sampling_frequency())/60))
    print("Data dtype:", RAW_RECORDING.get_dtype())

print("------------------------------------------------------------------------------------------------")

### **1.2**  Check noise level

In [ ]:
if 'PROC_RECORDING' not in locals():
    print("Step 1.2: Plot noise distributions...........................")
    plot_noise_hists(RAW_RECORDING)

### **1.3**  Filter recording and detect (and remove) bad channels
* Apply bandpass filter (highpass cutoff freq = 300 Hz, lowpass cutoff freq = 6000 Hz)
* Detect bad channels ("bad" = dead channel with low similarity to surrounding channels or to median of all channels, noise channels with power >80% Nyquist above 0.02 uV^2 /Hz)
* Apply phase shift to cancel the small sampling delay across recording system (margin = 40ms, ensures very small error when doing chunk processing)
* Re-reference traces by subtracting the 'global' median from all channels (common median reference, cmr) 

In [ ]:
# Apply filters and visualize steps
if 'PROC_RECORDING' not in locals():
    print("Step 1.3: Apply filters to recording...........................")
    FILT_RECORDING, bad_channels = preprocess_raw_recording(RAW_RECORDING, remove_bad_channels=remove_bad_channels)
    print("Bad channels:", bad_channels)
    print("------------------------------------------------------------------------------------------------")

### **1.4**  Detect and localize peaks

In [ ]:
#if 'PROC_RECORDING' not in locals():
    #plot_peaks_from_recording(FILT_RECORDING)

### **1.5**  Estimate motion/drift of probe
Calculates rough estimate of drift, to determine if the recording session needs to be chopped off somewhere from monkey shaking

In [ ]:
# Start with a fast estimation of drift on probe, useful for chopping out section of recording where monkey may have shook
#if 'PROC_RECORDING' not in locals():
#    motion, motion_info = si.compute_motion(FILT_RECORDING, preset='rigid_fast', output_motion_info=True)
#
#    plot_motion_correction(rec, motion_info)

### **1.6**  Correct motion/drift of probe

In [ ]:
if 'PROC_RECORDING' not in locals():
    if drift_correct.lower() != 'none':
        motion_folder = preprocess_folder / 'motion'
    
        if not Path(motion_folder).exists():
        if motion_folder.exists() and motion_folder.is_dir():
            shutil.rmtree(motion_folder)
            
        print("Step 1.6: Estimate motion from recording...........................")
        PROC_RECORDING, motion_info = si.correct_motion(FILT_RECORDING.astype(float), preset=drift_correct, interpolate_motion_kwargs={'border_mode':'force_extrapolate'}, folder=motion_folder, output_motion_info=True)
        PROC_RECORDING = PROC_RECORDING.astype(int)
    
        print("------------------------------------------------------------------------------------------------")
    else:
        rec = FILT_RECORDING.astype(float)
        PROC_RECORDING = rec.astype(int)
        

### **1.7**  Save preprocessed data to binary file

In [ ]:
PROC_RECORDING = PROC_RECORDING.astype(int)

if overwrite_preprocess:
    print("Step 1.7: Saving drift-corrected recording to disk...........................")
    PROC_RECORDING = PROC_RECORDING.save(folder=preprocess_folder / 'output', format='binary', dtype='int16', overwrite=True)

## **2. Spike sort the recording**
### **2.1**  Check if spike sorting has already been done: if so, load it; otherwise, continue on to spike sorting.

In [ ]:
if os.listdir(sorter_folder) and not overwrite_sorting and 'RAW_RECORDING' not in locals():
    print("Step 2.1: Loading sorting outputs...........................")
    SORTING = si.read_sorter_folder(sorter_folder)

In [ ]:
#REC_SLICE = RECORDING.frame_slice(start_frame=0,
#                                  end_frame=int(900*fs))

if 'SORTING' not in locals():
    sorting = si.run_sorter(sorter_type, PROC_RECORDING, remove_existing_folder=True, folder=sorter_folder,
                            docker_image=False, verbose=True, **custom_params)

In [ ]:
# 3. Postprocessing

In [ ]:
#ANALYZER = si.load_sorting_analyzer(folder=sorter_folder / "analyzer")

ANALYZER = si.create_sorting_analyzer(SORTING, PROC_RECORDING, sparse=True, format="memory") #"binary_folder", folder=sorter_folder / "analyzer")
ANALYZER

In [ ]:
ANALYZER.compute("isi_histograms")
ANALYZER.compute("correlograms")
ANALYZER.compute("acgs_3d")
ANALYZER.compute("noise_levels")
ANALYZER.compute("random_spikes", method="uniform", max_spikes_per_unit=500)

ANALYZER.compute("waveforms",  ms_before=1.5,ms_after=2., **job_kwargs)
ANALYZER.compute("templates", operators=["average", "median", "std"])

ANALYZER.compute("amplitude_scalings")
ANALYZER.compute("spike_amplitudes", **job_kwargs)
ANALYZER.compute("spike_locations", **job_kwargs)
ANALYZER.compute("template_metrics", **job_kwargs)
ANALYZER.compute("template_similarity")
ANALYZER.compute("unit_locations")

ANALYZER

In [ ]:
analyzer_saved = ANALYZER.save_as(folder=sorter_folder / "analyzer", format="binary_folder")
analyzer_saved

In [ ]:
# 4. Quality metrics 

In [ ]:
metric_names=['amplitude_cutoff','amplitude_cv_median','amplitude_cv_range','amplitude_median',
              'd_prime','drift_ptp','drift_std','drift_mad','firing_range','firing_rate','isi_violation','rp_violation',
              'isolation_distance','l_ratio','nn_hit_rate','nn_miss_rate','nn_isolation','nn_noise_overlap','presence_ratio',
              'sd_ratio','silhouette','silhouette_full','sliding_rp_violations','snr','synchrony']

METRICS = si.compute_quality_metrics(ANALYZER, metric_names=metric_names)
METRICS